In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Load dataset directly from IBM's public GitHub repo
df = pd.read_csv('https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv')

# Drop non-informative columns
df = df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber'])

# Encode target
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Encode categorical features
cat_cols = df.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

X = df.drop(columns=['Attrition'])
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Scale for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model 1: Logistic Regression
logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train_scaled, y_train)
pred_lr = logreg.predict(X_test_scaled)

print("Logistic Regression:")
print("  Accuracy:", round(accuracy_score(y_test, pred_lr), 3))
print("  Precision:", round(precision_score(y_test, pred_lr), 3))
print("  Recall:", round(recall_score(y_test, pred_lr), 3))
print("  F1-score:", round(f1_score(y_test, pred_lr), 3))

# Model 2: Decision Tree
dtree = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dtree.fit(X_train, y_train)
pred_dt = dtree.predict(X_test)

print("\nDecision Tree:")
print("  Accuracy:", round(accuracy_score(y_test, pred_dt), 3))
print("  Precision:", round(precision_score(y_test, pred_dt), 3))
print("  Recall:", round(recall_score(y_test, pred_dt), 3))
print("  F1-score:", round(f1_score(y_test, pred_dt), 3))

# Top features for each model
coef_df = pd.DataFrame({'feature': X.columns, 'coef': logreg.coef_[0]})
coef_df['abs_coef'] = coef_df['coef'].abs()
print("\nTop Logistic Regression features:")
print(coef_df.sort_values('abs_coef', ascending=False).head(8)[['feature','coef']])

feat_imp = pd.DataFrame({'feature': X.columns, 'importance': dtree.feature_importances_})
print("\nTop Decision Tree features:")
print(feat_imp.sort_values('importance', ascending=False).head(8))

Logistic Regression:
  Accuracy: 0.761
  Precision: 0.367
  Recall: 0.678
  F1-score: 0.476

Decision Tree:
  Accuracy: 0.766
  Precision: 0.329
  Recall: 0.441
  F1-score: 0.377

Top Logistic Regression features:
                    feature      coef
18                 OverTime  0.723402
29     YearsWithCurrManager -0.517110
3                Department  0.473786
15            MonthlyIncome -0.469469
28  YearsSinceLastPromotion  0.467772
23        TotalWorkingYears -0.430368
17       NumCompaniesWorked  0.425483
7   EnvironmentSatisfaction -0.415417

Top Decision Tree features:
               feature  importance
0                  Age    0.195509
26      YearsAtCompany    0.190156
18            OverTime    0.123861
17  NumCompaniesWorked    0.111844
4     DistanceFromHome    0.076689
25     WorkLifeBalance    0.049802
11            JobLevel    0.046862
3           Department    0.038053
